# 01 · WikiText-2 data preparation
**Goal:** train a SentencePiece BPE tokenizer on training text, use a 30,000-piece vocabulary, encode up to 3 million training subword tokens, and inspect split coverage. This replaces the old word-level tokenizer.


In [11]:
# Set project paths so notebook cells can import the shared src modules and find processed data.
from pathlib import Path
import sys, json, math, time
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data' / 'processed_3m_max_tokens_30k_vocab'
print('Project:', ROOT)

Project: /Users/paulng/Y4S1 Local files/DSA4213/dsa4213_oct13


## Configuration
The tokenizer is trained on the WikiText-2 training split only. The training stream is capped at 3 million subword IDs; validation and test remain complete.


In [12]:
# Requested Part I settings: 30,000 BPE pieces and up to 3 million train tokens.
VOCAB_SIZE = 30_000
MAX_TRAIN_TOKENS = 3_000_000
USE_DEMO = False  # True only for a quick offline pipeline check
SEED = 42


In [13]:
# Load raw WikiText-2 splits (or demo text); inspect examples before tokenizing.
if USE_DEMO:
    lines = ['The scientist studied the language model.', 'The model learned from text.'] * 200
    splits = {'train': lines, 'validation': lines[:40], 'test': lines[40:80]}
else:
    from datasets import load_dataset
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')
    splits = {split: ds[split]['text'] for split in ('train', 'validation', 'test')}
print({split: len(lines) for split, lines in splits.items()})


{'train': 36718, 'validation': 3760, 'test': 4358}


In [14]:
# Show 10 content-bearing raw lines from each split; the dataset page itself only previews a few.
import random
EXAMPLES_PER_SPLIT = 10 # increase this to see more lines
rng = random.Random(SEED)
for split, lines in splits.items():
    candidates = [(i, line.strip()) for i, line in enumerate(lines) if len(line.strip()) >= 60]
    selected = rng.sample(candidates, min(EXAMPLES_PER_SPLIT, len(candidates)))
    print(f"\n{split.upper()} — {len(lines):,} source rows; showing {len(selected)} examples")
    for index, line in selected:
        print(f"[{index}] {line}")



TRAIN — 36,718 source rows; showing 10 examples
[24709] The route continues on , crossing over Owego Creek and passing the Newark Valley Country Club about 1 mile ( 1 @.@ 6 km ) north of Newark Valley village before entering the town of Berkshire . In Berkshire , NY 38 serves the hamlet of Berkshire , a small community situated directly on the highway . The route continues on through the narrowing creek valley into the town of Richford and the hamlet of the same name , where it meets NY 79 in the community 's center . After Richford , the valley continues to narrow for just under 1 @.@ 5 miles ( 2 @.@ 4 km ) before reversing course as the route heads into Cortland County and the town of Harford .
[4424] Galveston is the setting of Sean Stewart 's 2000 fantasy novel Galveston , in which a Flood of Magic takes over the island city , resulting in strange and carnivalesque adventures . It tied in 2001 with Declare , by Tim Powers , for the World Fantasy Award for Best Novel . It also won 

In [15]:
# Train SentencePiece BPE on the training lines and replace processed_3m_max_tokens_30k_vocab in place.
from data import prepare
prepare(splits, DATA, vocab_size=VOCAB_SIZE, max_train_tokens=MAX_TRAIN_TOKENS, seed=SEED)


I0000 00:00:1792045352.271739 42141689 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: /Users/paulng/Y4S1 Local files/DSA4213/dsa4213_oct13/data/processed_3m_max_tokens_30k_vocab/tokenizer
  model_type: BPE
  vocab_size: 30000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 0
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 16384
  num_threads: 1
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: -1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  e

train: 2,224,073 subword tokens; UNK 0.000%
validation: 239,175 subword tokens; UNK 0.000%
test: 272,269 subword tokens; UNK 0.000%


In [16]:
# Inspect token IDs, target counts, byte-fallback unknown rates, and example BPE pieces.
from collections import Counter
vocab = json.loads((DATA/'vocab.json').read_text())
ids = {s: json.loads((DATA/f'{s}.json').read_text()) for s in ('train','validation','test')}
for split, seq in ids.items():
    print(f'{split:10} subword token IDs={len(seq):>9,}  UNK={seq.count(1)/max(1,len(seq)):.3%}')
print('\nNext-token target counts:')
print(f"train per full epoch: {len(ids['train'])-1:,} (IDs minus the first context token)")
print(f"validation: {len(ids['validation'])-1:,} (IDs minus the first context token)")
print(f"common test: {len(ids['test'])-2:,} (positions 2 onward; two initial IDs provide context for the trigram)")
print('Vocabulary size:', len(vocab))
print('First 30 pieces:', [vocab[i] for i in ids['train'][:30]])

train      subword token IDs=2,224,073  UNK=0.000%
validation subword token IDs=  239,175  UNK=0.000%
test       subword token IDs=  272,269  UNK=0.000%

Next-token target counts:
train per full epoch: 2,224,072 (IDs minus the first context token)
validation: 239,174 (IDs minus the first context token)
common test: 272,267 (positions 2 onward; two initial IDs provide context for the trigram)
Vocabulary size: 30000
First 30 pieces: ['▁=', '▁valkyria', '▁chronicles', '▁iii', '▁=', '<eos>', '▁sen', 'jō', '▁no', '▁valkyria', '▁3', '▁:', '▁unrec', 'orded', '▁chronicles', '▁(', '▁japanese', '▁:', '▁', '戦', '場', 'の', 'ヴ', 'ァ', 'ル', 'キ', 'ュ', 'リ', 'ア', '3']
